# Augmented crops for the cluster name classifier

Makes **extra crops** by shifting, zooming, rotating, flipping and changing the lighting of the labeled
crops, and embeds them with the same fine-tuned BioCLIP. Two modes (`MODE` in Settings):
- **`"all"`**: every crop gets `COPIES_PER_CROP` copies. Class balance stays the same, so this tests whether
  augmentation itself helps.
- **`"rare"`**: only classes with fewer than `TARGET_PER_CLASS` crops, topped up to about that many. Tried
  2026-11-10: rare classes +1 point, common classes −3 points, overall −1 point (it re-tilts training toward rare classes).

1. finds the labeled crops on Drive (same as the `labeled_crops_eval` notebook)
2. picks which crops get augmented and how many copies each gets
3. checks that re-embedding an original crop here reproduces its stored embedding (same model + preprocessing)
4. makes and embeds the augmented crops
5. saves `augmented_labels.csv` + `augmented_embeddings.json`

**Before you start:** T4 GPU · a shortcut to **NatGeo Pristine Seas - FA26** in My Drive ·
`Matewos's Notebook/checkpoints/best.pt`. Nothing in the shared folder is changed.

**Afterwards (locally):** put the two files in `checkpoints/augmented_experiment/<MODE>/` and run
`python eval_classifier.py --aug-labels checkpoints/augmented_experiment/all/augmented_labels.csv --aug-embeddings checkpoints/augmented_experiment/all/augmented_embeddings.json`
(add `--max-copies 1` or `2` to test fewer copies without rerunning this notebook).
Only files in `checkpoints/labeled_crops/` are used by the app's retraining, so move them there once they're a win.
Augmented crops are training data only: they never count toward the 5-crop minimum and are never test crops.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: works, but embedding is slow")

## 1. Settings

In [ ]:
from pathlib import Path
import glob, json, math, os, shutil, sys
import numpy as np
import pandas as pd

MY_DIR = Path("/content/drive/MyDrive/Matewos's Notebook")
BRANCH = "feature/cluster-classifier-suggestions"   # "main" once it's merged

MODE = "all"             # "all": every crop gets COPIES_PER_CROP copies · "rare": top up the rare classes
COPIES_PER_CROP   = 3    # MODE="all" (test 1 or 2 later with eval_classifier.py --max-copies)
TARGET_PER_CLASS  = 50   # MODE="rare": top classes up to about this many crops (originals + augmented)
MAX_AUG_PER_CROP  = 10   # MODE="rare": never more copies than this of one crop (they're still one animal)
MIN_PER_CLASS     = 5    # MODE="rare": classes with fewer ORIGINAL crops aren't model classes, so they're skipped
SEED = 0
OUT = MY_DIR / "results" / "labeled_crops" / f"augmented_{MODE}"

hits = glob.glob("/content/drive/MyDrive/**/LABELED_CROPS", recursive=True)
assert hits, "LABELED_CROPS not found: add a shortcut to 'NatGeo Pristine Seas - FA26' in My Drive"
LC = Path(hits[0])
print("LABELED_CROPS:", LC)

## 2. Code, model weights and the labeled crops

In [ ]:
!rm -rf /content/nat-geo-fa26
!git clone -q --depth 1 -b {BRANCH} https://github.com/LaurenMaiH/nat-geo-fa26.git /content/nat-geo-fa26
!pip -q install -r /content/nat-geo-fa26/requirements.txt
REPO = Path("/content/nat-geo-fa26")
CKPT_DRIVE = MY_DIR / "checkpoints" / "best.pt"
if CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, REPO / "checkpoints" / "best.pt")
    print("fine-tuned best.pt copied from Drive")

labels = pd.read_csv(REPO / "checkpoints" / "labeled_crops" / "image_labels.csv")
stored = json.loads((REPO / "checkpoints" / "labeled_crops" / "images_embeddings.json").read_text())
print(f"{len(labels)} labeled crops in the repo")

In [ ]:
# Each mission's crops_manifest.csv says where its crop images are (paths were written on a
# teammate's laptop, so each crop is found by the tail of its path inside the mission folder).
def resolve(p, root):
    parts = [x for x in str(p).replace("\\", "/").split("/") if x]
    for n in range(len(parts), 0, -1):
        cand = root / "/".join(parts[-n:])
        if cand.exists():
            return str(cand)
    return None

parts = []
for f in sorted(LC.glob("*/crops_manifest.csv")):
    m = pd.read_csv(f)
    if "crop_status" in m:
        m = m[m["crop_status"].astype(str) == "cropped"]
    m["crop_path"] = [resolve(p, f.parent) for p in m["crop_path"]]
    parts.append(m)
man = pd.concat(parts, ignore_index=True).dropna(subset=["crop_path"]).drop_duplicates(["deployment", "row"])
man["key"] = man["deployment"].astype(str) + "__row" + man["row"].astype(int).astype(str)
path_of = dict(zip(man["key"], man["crop_path"]))

found = labels["key"].isin(path_of)
print(f"crop image found for {found.sum()} / {len(labels)} labeled crops")
assert found.mean() > 0.9, f"keys don't line up; e.g. labels {labels['key'].head(3).tolist()} vs manifest {man['key'].head(3).tolist()}"

## 3. Which crops get augmented, and how many times
- `MODE="all"`: every crop gets `COPIES_PER_CROP` copies.
- `MODE="rare"`: a class is a crop's genus, or its family when it has no genus (the names the model learns). Each crop of a
  class with `n` crops (5 ≤ n < `TARGET_PER_CLASS`) gets `ceil((TARGET − n) / n)` copies, at most `MAX_AUG_PER_CROP`.

In [ ]:
labels["name"] = [f if pd.isna(g) or str(g).strip() == "" else g for g, f in zip(labels["genus"], labels["family"])]
n_class = labels["name"].value_counts()
if MODE == "all":
    labels["n_aug"] = [COPIES_PER_CROP if ok else 0 for ok in found]
elif MODE == "rare":
    labels["n_aug"] = [0 if not (MIN_PER_CLASS <= n_class[c] < TARGET_PER_CLASS) or not ok
                       else min(MAX_AUG_PER_CROP, math.ceil((TARGET_PER_CLASS - n_class[c]) / n_class[c]))
                       for c, ok in zip(labels["name"], found)]
else:
    raise ValueError(f"MODE must be 'all' or 'rare', not {MODE!r}")
plan = labels[labels["n_aug"] > 0].groupby("name").agg(crops=("key", "size"), copies_each=("n_aug", "first"),
                                                        new_crops=("n_aug", "sum")).sort_values("crops")
print(plan.to_string() if len(plan) <= 40 else plan.sort_values("crops", ascending=False).head(20).to_string() + "\n...")
print(f"\n{len(plan)} classes, {plan['new_crops'].sum()} augmented crops to make")

## 4. The augmentation
Each copy gets a random combination of:
- **rotation** ±15° (on a mirrored border, so no black corners)
- **shift + zoom in**: a box 75–100% of the crop's size, moved up to 10%. This mimics the detector's less tidy boxes.
- **horizontal flip** (half the time)
- **brightness / contrast** ±20%, **saturation** −20% to +10%. There's no hue change, because colour is a clue (red shrimp).
- **blur** (30%) and **sensor noise** (30%): murky water, a distant animal

There are no vertical flips or big rotations, because fish and sharks have an "up". Check the preview: every copy should
still clearly be the animal.

In [ ]:
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

def augment(img, rng):
    img = img.convert("RGB")
    w, h = img.size
    a = np.asarray(img)
    pad = int(0.25 * max(w, h))
    big = Image.fromarray(np.pad(a, ((pad, pad), (pad, pad), (0, 0)), mode="reflect"))
    angle = rng.uniform(-15, 15)
    big = big.rotate(angle, resample=Image.BICUBIC)
    s = rng.uniform(0.75, 1.0)
    dx, dy = rng.uniform(-0.1, 0.1) * w, rng.uniform(-0.1, 0.1) * h
    cx, cy = pad + w / 2 + dx, pad + h / 2 + dy
    out = big.crop((cx - s * w / 2, cy - s * h / 2, cx + s * w / 2, cy + s * h / 2)).resize((w, h), Image.BICUBIC)
    flip = rng.random() < 0.5
    if flip:
        out = ImageOps.mirror(out)
    b, c, sat = rng.uniform(0.8, 1.2), rng.uniform(0.8, 1.2), rng.uniform(0.8, 1.1)
    out = ImageEnhance.Color(ImageEnhance.Contrast(ImageEnhance.Brightness(out).enhance(b)).enhance(c)).enhance(sat)
    blur = rng.uniform(0.5, 1.5) if rng.random() < 0.3 else 0
    if blur:
        out = out.filter(ImageFilter.GaussianBlur(blur))
    noise = rng.uniform(3, 10) if rng.random() < 0.3 else 0
    if noise:
        arr = np.asarray(out).astype(np.float32) + rng.normal(0, noise, (h, w, 3))
        out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    params = f"rot={angle:.0f} zoom={s:.2f} shift=({dx / w:.2f},{dy / h:.2f}) flip={int(flip)} " \
             f"bright={b:.2f} contrast={c:.2f} sat={sat:.2f} blur={blur:.1f} noise={noise:.0f}"
    return out, params

# Preview: 6 rare-class crops, original first, then 5 copies
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
sample = labels[labels["n_aug"] > 0].sample(6, random_state=SEED)
fig, axes = plt.subplots(6, 6, figsize=(12, 12))
for row, r in zip(axes, sample.itertuples()):
    img = Image.open(path_of[r.key])
    row[0].imshow(img); row[0].set_title(r.name, fontsize=8)
    for ax in row[1:]:
        ax.imshow(augment(img, rng)[0])
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Load the fine-tuned BioCLIP, and check it reproduces the stored embeddings
Re-embeds 32 original crops the way the pipeline does and compares them with the stored embeddings.
Similarity should be ~1.00. If it isn't, the augmented crops would be embedded differently from the originals and
the comparison would be meaningless, so the notebook stops.

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import cv2
import overnight_pipeline_optimized as pipe
pipe.INFERENCE_DEVICE = pipe.resolve_device(None)
embed_info = pipe.load_embedding_model()
assert embed_info[0] == "finetuned", f"BioCLIP weights: {embed_info[0]}; need the fine-tuned best.pt"

def embed(pil_images, batch=64):
    out = []
    for i in range(0, len(pil_images), batch):
        bgr = [cv2.cvtColor(np.asarray(im.convert("RGB")), cv2.COLOR_RGB2BGR) for im in pil_images[i:i + batch]]
        out.append(pipe.get_embeddings_batch(embed_info, bgr))
    return np.concatenate(out)

check = labels[found].sample(32, random_state=SEED)
mine = embed([Image.open(path_of[k]) for k in check["key"]])
theirs = np.asarray([stored[k] for k in check["key"]], dtype=np.float32)
theirs /= np.linalg.norm(theirs, axis=1, keepdims=True)
sim = (mine * theirs).sum(axis=1)
print(f"similarity to stored embeddings: min {sim.min():.4f}, median {np.median(sim):.4f}")
assert sim.min() > 0.98, "re-embedded crops don't match the stored embeddings: different model or preprocessing"

## 6. Make and embed the augmented crops
A few minutes on a T4.

In [ ]:
from tqdm.auto import tqdm
rng = np.random.default_rng(SEED)
todo = labels[labels["n_aug"] > 0]
keep_cols = [c for c in labels.columns if c not in ("name", "n_aug")]
rows, aug_emb = [], {}
pending_imgs, pending_keys = [], []

def flush():
    if pending_imgs:
        for k, v in zip(pending_keys, embed(pending_imgs)):
            aug_emb[k] = [round(float(x), 6) for x in v]
        pending_imgs.clear(); pending_keys.clear()

# Plain dicts, not itertuples(): a column named "class" (a Python keyword) gets renamed there
for r in tqdm(todo.to_dict("records"), total=len(todo)):
    img = Image.open(path_of[r["key"]]).convert("RGB")
    for i in range(r["n_aug"]):
        im, params = augment(img, rng)
        k = f"{r['key']}__aug{i}"
        row = {c: r[c] for c in keep_cols}
        row.update(key=k, augmented_from=r["key"], augmentation=params)
        rows.append(row)
        pending_imgs.append(im); pending_keys.append(k)
        if len(pending_imgs) >= 256:
            flush()
flush()
aug_labels = pd.DataFrame(rows)
print(f"{len(aug_labels)} augmented crops of {aug_labels['augmented_from'].nunique()} originals")

## 7. Save

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
aug_labels.to_csv(OUT / "augmented_labels.csv", index=False)
(OUT / "augmented_embeddings.json").write_text(json.dumps(aug_emb))
print("saved to", OUT)
print(f"Download both files into checkpoints/augmented_experiment/{MODE}/ in the repo, then run eval_classifier.py (see the top).")